In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# ----- 1) Imports & Reproducibility -----
import os
import math
import random
import numpy as np
import pandas as pd
import re

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

In [ ]:
Water_Quality_df = pd.read_csv("data/water_quality_training_dataset.csv")
landsat_train_features = pd.read_csv("data/landsat/landsat_features_training_mvdb.csv")
Terraclimate_df = pd.read_csv("data/terraclimate_features_training_combined.csv")

#Clean Up the Data
#Capitalize Col Names
def capitalize_words_keep_spacing(col):
    # Split on space or underscore but keep the separators
    parts = re.split(r'([ _])', col)
    # Capitalize text parts, keep separators unchanged
    return ''.join(p.title() if p not in [' ', '_'] else p for p in parts)
Terraclimate_df.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df.columns]
landsat_train_features.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features.columns]
Water_Quality_df.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df.columns]

def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)

#Data type corrections 
wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'],  format='%d-%m-%Y')


def convert_text_cols_to_float(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if (pd.api.types.is_object_dtype(out[c]) 
            or pd.api.types.is_string_dtype(out[c]) 
            or pd.api.types.is_categorical_dtype(out[c])):
            s = out[c].astype(str).str.replace('\u00A0', ' ', regex=False).str.strip()
            s = s.str.replace('\u2212', '-', regex=False)                 # unicode minus
            s = s.str.replace(r'^\(\s*(.*)\s*\)$', r'-\1', regex=True)    # (123) -> -123
            s = s.str.replace(r'^\s*([+]?\s*[\d, .]+)\s*-$', r'-\1', regex=True)  # 123- -> -123
            s = s.str.replace(r'(?<=\d),(?=\d{3}\b)', '', regex=True)      # thousands comma
            out[c] = pd.to_numeric(s, errors='coerce')                     # float64 by default with NaNs
    return out

wq_data = convert_text_cols_to_float(wq_data)

#ullify all negative observations
for column in wq_data.columns:
    if column != "Sample Date": wq_data[wq_data[column] < -9000][column] = np.nan 

wq_data['Month_cosine'] = np.cos((wq_data['Sample Date'].dt.month + (wq_data['Sample Date'].dt.day/31))* np.pi / 6)

wq_data = wq_data.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Sample Date',  'Atmos_Opacity', 'Emsd', 'Lwir', 'Unnamed: 0'])
#, 'Atmos opacity, 'Emsd', 'Lwir''
feature_cols = wq_data.columns.tolist()

#print(wq_data.columns)
feature_cols.remove('Latitude')
feature_cols.remove('Longitude')
feature_cols.remove('Total Alkalinity')
feature_cols.remove('Electrical Conductance')
feature_cols.remove('Dissolved Reactive Phosphorus')

wq_data = wq_data.dropna(subset=feature_cols)



In [ ]:
Water_Quality_df_v = pd.read_csv("data/submission_template.csv")
landsat_train_features_v = pd.read_csv("data/landsat/landsat_features_validation_mvdb.csv")
Terraclimate_df_v = pd.read_csv("data/terraclimate_features_validation_combined.csv")

#Clean Up the Data
Terraclimate_df_v.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df_v.columns]
landsat_train_features_v.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features_v.columns]
Water_Quality_df_v.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df_v.columns]

wq_data_v = combine_two_datasets(Water_Quality_df_v, landsat_train_features_v, Terraclimate_df_v)

#Data type corrections 
wq_data_v['Sample Date'] = pd.to_datetime(wq_data_v['Sample Date'],  format='%d-%m-%Y')
wq_data_v = convert_text_cols_to_float(wq_data_v)

wq_data_v = wq_data_v.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Atmos_Opacity', 'Emsd', 'Lwir', 'Unnamed: 0'])
#ullify all negative observations
for column in wq_data_v.columns:
    if column != "Sample Date": wq_data_v[wq_data_v[column] < -9000][column] = np.nan 

wq_data_v['Month_cosine'] = np.cos((wq_data_v['Sample Date'].dt.month + (wq_data_v['Sample Date'].dt.day/31))* np.pi / 6)

wq_data_v = wq_data_v.drop(columns='Sample Date')

wq_data_v = wq_data_v.dropna(subset=feature_cols)

wq_data_v = wq_data_v[sorted(wq_data_v.columns)]

In [ ]:
wq_data_v.shape

In [ ]:
#number of cv groups
cv_groups = 8

#split over longitude
wq_data['cv_group'] = pd.qcut(wq_data['Latitude'], q=cv_groups, labels=False)

# Specify the number of folds
lat_sep_kf = GroupKFold(n_splits=cv_groups - 1)

In [ ]:
#test train based on location
wq_data_test = wq_data[wq_data['cv_group'] == 4]
wq_data_train = wq_data[wq_data['cv_group'] != 4]

wq_data_test = wq_data_test.drop(columns='cv_group')
wq_data_train = wq_data_train.drop(columns='cv_group')

wq_data_train = wq_data_train[sorted(wq_data_train.columns)]
wq_data_test = wq_data_test[sorted(wq_data_test.columns)]

#then split into X and Y 
#Y_train = wq_data_train[["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]]
#X_train = wq_data_train.drop(columns=["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus", "Longitude", "Latitude"])
#Y_test = wq_data_test[["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]]
#X_test = wq_data_test.drop(columns=["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"])

In [ ]:
# For reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)



In [ ]:
# ----- 2) Configurable column names & task type -----
# Adjust these lists for your real dataset
key_cols     = ["Latitude", "Longitude"]  # 3 key columns (IDs, timestamps, etc.)
feature_cols = wq_data.columns.tolist()
feature_cols.remove('Latitude')
feature_cols.remove('Longitude')
feature_cols.remove('Total Alkalinity')
feature_cols.remove('Electrical Conductance')
feature_cols.remove('Dissolved Reactive Phosphorus')
feature_cols.remove('cv_group')
#feature_cols = [f"f{i}" for i in range(1, 21)]  # 20 feature columns
target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
#target_cols  = ["y1", "y2", "y3"]  # 3 target columns

# Choose the task: "regression" (default) or "multilabel"
#  - regression:  continuous y1,y2,y3  -> MSELoss
#  - multilabel:  binary y1,y2,y3 in {0,1} -> BCEWithLogitsLoss
task_type = "regression"


In [ ]:
# # ----- 3) Create a synthetic dataset (replace this block with pd.read_csv for your data) -----
# N = 5000  # number of rows

# # Create keys (not used as features, but kept to reattach predictions later)
# df = pd.DataFrame({
#     "Latitude": np.arange(wq_data),
#     "Longitude": np.random.choice(list("ABCDE"), size=N),
# })

# # Create 20 numeric features
# for i in range(1, 21):
#     df[f"f{i}"] = np.random.randn(N) * (0.5 + i * 0.05) + i  # different scales

# # Create 3 targets:
# #   - For regression, continuous targets are noisy combos of features.
# #   - For multilabel (if you switch), we threshold them later.
# y1 = 0.8*df["f1"] - 0.3*df["f2"] + 0.5*df["f3"] + 0.1*np.random.randn(N)
# y2 = -0.2*df["f4"] + 1.1*df["f5"] + 0.1*df["f6"] + 0.1*np.random.randn(N)
# y3 = 0.4*df["f7"] + 0.2*df["f8"] - 0.5*df["f9"] + 0.1*np.random.randn(N)

# if task_type == "regression":
#     df["y1"], df["y2"], df["y3"] = y1, y2, y3
# else:  # multilabel example targets (0/1)
#     df["y1"] = (y1 > y1.mean()).astype(int)
#     df["y2"] = (y2 > y2.mean()).astype(int)
#     df["y3"] = (y3 > y3.mean()).astype(int)

In [ ]:
# # ----- 4) Basic cleaning: keep only needed columns, handle NaNs -----
# needed_cols = key_cols + feature_cols + target_cols
# df = df[needed_cols].copy()
# df = df.fillna(0)


In [ ]:
# # ----- 5) Train / Val / Test Split (70 / 15 / 15) -----
# indices = np.arange(len(df))
# np.random.shuffle(indices)

# n_train = int(0.70 * len(df))
# n_val   = int(0.15 * len(df))
# train_idx = indices[:n_train]
# val_idx   = indices[n_train:n_train + n_val]
# test_idx  = indices[n_train + n_val:]

# df_train = df.iloc[train_idx].reset_index(drop=True)
# df_val   = df.iloc[val_idx].reset_index(drop=True)
# df_test  = df.iloc[test_idx].reset_index(drop=True)


In [ ]:
print(wq_data_train.shape)
print(wq_data_test.shape)
print(wq_data_v.shape)

In [ ]:
print(wq_data_train.columns)
print(wq_data_test.columns)
print(wq_data_v.columns)

In [ ]:
wq_data_test.info()

In [ ]:
# ----- 6) Compute normalization stats on TRAIN ONLY -----
# Standardize features to zero-mean, unit-std
feat_means = wq_data_train[feature_cols].mean()
feat_stds  = wq_data_train[feature_cols].std().replace(0, 1.0)  # avoid division by zero

def normalize_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out[feature_cols] = (out[feature_cols] - feat_means) / feat_stds
    return out

#scaler = StandardScaler()
#df_train_n = scaler.fit_transform(wq_data_train)
#df_val_n = scaler.transform(wq_data_test)
#df_test_n = scaler.transform(wq_data_v)



df_train_n = normalize_features(wq_data_train)
df_val_n   = normalize_features(wq_data_test)
df_test_n  = normalize_features(wq_data_v)

# (Optional) You could also normalize targets for regression if scales differ greatly.
# For simplicity, we leave targets as-is.




In [ ]:
# print(df_train_n.shape)
# print(df_val_n.shape)
# print(df_test_n.shape)

In [ ]:
# df_test_n.head(5)

In [ ]:
# ----- 7) PyTorch Dataset -----
class TabularDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, key_cols, feature_cols, target_cols, task_type="regression"):
        self.keys = frame[key_cols].reset_index(drop=True)
        X = frame[feature_cols].to_numpy(dtype=np.float32)
        self.X = torch.from_numpy(X)
        y = frame[target_cols].to_numpy(dtype=np.float32)
        self.y = torch.from_numpy(y)
        self.task_type = task_type

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Return keys (as dict-like), features, and targets
        return self.keys.iloc[idx].to_dict(), self.X[idx], self.y[idx]

train_ds = TabularDataset(df_train_n, key_cols, feature_cols, target_cols, task_type=task_type)
val_ds   = TabularDataset(df_val_n,   key_cols, feature_cols, target_cols, task_type=task_type)
test_ds  = TabularDataset(df_test_n,  key_cols, feature_cols, target_cols, task_type=task_type)



In [ ]:
# ----- 8) DataLoaders -----
BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [ ]:
# ----- 9) Model: a simple MLP -----
class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden_dims=(128, 64), dropout=0.1, task_type="regression"):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)
        self.task_type = task_type

    def forward(self, x):
        logits_or_outputs = self.net(x)
        # For regression, return raw outputs.
        # For multilabel classification: during training we keep raw logits (BCEWithLogitsLoss expects logits).
        return logits_or_outputs

model = MLP(in_dim=len(feature_cols), out_dim=len(target_cols), hidden_dims=(128, 64), dropout=0.15, task_type=task_type).to(DEVICE)



In [ ]:
# ----- 10) Loss and Optimizer -----
if task_type == "regression":
    criterion = nn.MSELoss()
else:  # "multilabel"
    criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

In [ ]:
# ----- 11) Metrics -----
@torch.no_grad()
def evaluate(model, loader, task_type="regression", threshold=0.5):
    model.eval()
    total_loss = 0.0
    n_examples = 0

    # For regression: track MAE and RMSE
    mae_sum = torch.zeros(len(target_cols), device=DEVICE)
    mse_sum = torch.zeros(len(target_cols), device=DEVICE)

    # For multilabel: track accuracy (micro)
    correct = 0
    total   = 0

    for _, X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        out = model(X)

        loss = criterion(out, y)
        bs = y.size(0)
        total_loss += loss.item() * bs
        n_examples += bs

        if task_type == "regression":
            err = torch.abs(out - y)
            mae_sum += err.sum(dim=0)
            mse_sum += ((out - y) ** 2).sum(dim=0)
        else:
            # multilabel: apply sigmoid then threshold
            probs = torch.sigmoid(out)
            preds = (probs >= threshold).float()
            correct += (preds == y).sum().item()
            total   += y.numel()

    avg_loss = total_loss / max(n_examples, 1)

    if task_type == "regression":
        mae = (mae_sum / n_examples).detach().cpu().numpy()
        rmse = torch.sqrt(mse_sum / n_examples).detach().cpu().numpy()
        metrics = {
            "loss": avg_loss,
            "MAE_per_target": {t: float(mae[i]) for i, t in enumerate(target_cols)},
            "RMSE_per_target": {t: float(rmse[i]) for i, t in enumerate(target_cols)},
        }
    else:
        accuracy = correct / max(total, 1)
        metrics = {"loss": avg_loss, "micro_accuracy": accuracy}

    return metrics

In [ ]:
# ----- 12) Training Loop -----
EPOCHS = 30
best_val = math.inf
patience = 5
wait = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    n_seen = 0

    for _, X, y in train_loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        n_seen += X.size(0)

    train_loss = running_loss / max(n_seen, 1)
    val_metrics = evaluate(model, val_loader, task_type=task_type)
    val_loss = val_metrics["loss"]

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Metrics: {val_metrics}")

    # Simple early stopping on validation loss
    if val_loss < best_val - 1e-6:
        best_val = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping.")
            break

# Load best model (if available)
if best_state is not None:
    model.load_state_dict(best_state)

In [ ]:
# ----- 13) Final evaluation on TEST -----
test_metrics = evaluate(model, test_loader, task_type=task_type)
print("TEST:", test_metrics)


In [ ]:
# ----- 14) Inference: get predictions + keys side-by-side -----
@torch.no_grad()
def predict_with_keys(model, loader, task_type="regression", threshold=0.5):
    model.eval()
    all_rows = []
    for keys, X, _ in loader:
        X = X.to(DEVICE)
        logits_or_outputs = model(X).cpu().numpy()

        if task_type == "regression":
            preds = logits_or_outputs  # raw outputs
        else:
            probs = 1 / (1 + np.exp(-logits_or_outputs))  # sigmoid
            preds = (probs >= threshold).astype(np.float32)

        # For each batch row, combine keys + predictions into a dict
        for i in range(len(keys["Latitude"])):
            row = {k: keys[k][i] for k in keys.keys()}
            for j, t in enumerate(target_cols):
                row[f"pred_{t}"] = float(preds[i, j])
            all_rows.append(row)
    return pd.DataFrame(all_rows)

pred_df = predict_with_keys(model, test_loader, task_type=task_type)
pred_df.head(5)

In [ ]:
display(pred_df)

In [ ]:
# ----- 15) (Optional) Save artifacts -----
os.makedirs("artifacts", exist_ok=True)
torch.save({
    "model_state_dict": model.state_dict(),
    "feature_cols": feature_cols,
    "target_cols": target_cols,
    "feat_means": feat_means.to_dict(),
    "feat_stds": feat_stds.to_dict(),
    "task_type": task_type
}, "artifacts/model.pt")

pred_df.to_csv("artifacts/test_predictions_with_keys.csv", index=False)
print("Saved: artifacts/model.pt and artifacts/test_predictions_with_keys.csv")